In [1]:
from jetbot import Camera

camera = Camera.instance(
    width=224,
    height=224
)

print("Camera ready!")

Camera ready!


In [2]:
import torch
import torch.nn as nn

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

class SteeringCNN(nn.Module):
    def __init__(self):
        super(SteeringCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=5, stride=2),
            nn.ReLU(),

            nn.Conv2d(16, 32, kernel_size=5, stride=2),
            nn.ReLU(),

            nn.Conv2d(32, 64, kernel_size=3, stride=2),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((4, 4))
        )

        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.regressor(x)
        return x.squeeze(1)

print("SteeringCNN ready!")
print("Device:", device)

SteeringCNN ready!
Device: cuda


In [ ]:
old_model = SteeringCNN().to(device)

old_model.load_state_dict(
    torch.load(
        "models/clockwise_steering_best.pth",
        map_location=device
    )
)

old_model.eval()

print("OLD clockwise model loaded!")
print("526 samples")

In [3]:
new_model = SteeringCNN().to(device)

new_model.load_state_dict(
    torch.load(
        "models/clockwise_steering_1573_best.pth",
        map_location=device
    )
)

new_model.eval()

print("NEW clockwise model loaded!")
print("1573 samples")

NEW clockwise model loaded!
1573 samples


In [3]:
best_model = SteeringCNN().to(device)

best_model.load_state_dict(
    torch.load(
        "models/clockwise_best_model.pth",
        map_location=device
    )
)

best_model.eval()

print("BEST clockwise model loaded!")
print("Model: clockwise_best_model.pth")

BEST clockwise model loaded!
Model: clockwise_best_model.pth


In [3]:
best_model = SteeringCNN().to(device)

best_model.load_state_dict(
    torch.load(
        "models/clockwise_best_model_2868.pth",
        map_location=device
    )
)

best_model.eval()

print("BEST clockwise model loaded!")
print("Model: clockwise_best_model_2868.pth")

BEST clockwise model loaded!
Model: clockwise_best_model_2868.pth


In [3]:
best_model = SteeringCNN().to(device)

best_model.load_state_dict(
    torch.load(
        "models/clockwise_best_model_3361.pth",
        map_location=device
    )
)

best_model.eval()

print("3361 model loaded!")

3361 model loaded!


In [3]:
from jetbot import Robot, bgr8_to_jpeg
import torch
import time
import threading
import ipywidgets as widgets
import traitlets

from PIL import Image
from torchvision import transforms
from IPython.display import display


# ==========================================
# ROBOT
# ==========================================

robot = Robot()


# ==========================================
# AUTO SETTINGS
# ==========================================

AUTO_SPEED = 0.15

LEFT_MOTOR_GAIN = 1.035
RIGHT_MOTOR_GAIN = 1.00

STEERING_SCALE = 0.10
MAX_STEERING = 0.20

auto_running = False


# ==========================================
# IMAGE TRANSFORM
# ==========================================

live_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])


# ==========================================
# LIVE CAMERA
# ==========================================

camera_view = widgets.Image(
    format="jpeg",
    width=400,
    height=300
)

camera_link = traitlets.dlink(
    (camera, "value"),
    (camera_view, "value"),
    transform=bgr8_to_jpeg
)


# ==========================================
# DISPLAY
# ==========================================

prediction_label = widgets.Label(
    value="Steering: ---"
)

motor_label = widgets.Label(
    value="Motors: STOPPED"
)


# ==========================================
# MOTOR CONTROL
# ==========================================

def drive_from_prediction(prediction):

    steering = max(
        -MAX_STEERING,
        min(MAX_STEERING, prediction)
    )

    steering_power = steering * STEERING_SCALE

    left_speed = (
        AUTO_SPEED + steering_power
    ) * LEFT_MOTOR_GAIN

    right_speed = (
        AUTO_SPEED - steering_power
    ) * RIGHT_MOTOR_GAIN

    left_speed = max(0.0, min(1.0, left_speed))
    right_speed = max(0.0, min(1.0, right_speed))

    robot.left_motor.value = left_speed
    robot.right_motor.value = right_speed

    motor_label.value = (
        f"L: {left_speed:.3f} | "
        f"R: {right_speed:.3f}"
    )


# ==========================================
# AUTO LOOP
# ==========================================

def auto_loop():

    global auto_running

    while auto_running:

        image_bgr = camera.value

        if image_bgr is None:
            robot.stop()
            time.sleep(0.05)
            continue

        image_rgb = image_bgr[:, :, ::-1].copy()
        pil_image = Image.fromarray(image_rgb)

        input_tensor = (
            live_transform(pil_image)
            .unsqueeze(0)
            .to(device)
        )

        with torch.no_grad():
            steering_prediction = model(input_tensor).item()

        prediction_label.value = (
            f"Steering: {steering_prediction:.3f}"
        )

        drive_from_prediction(steering_prediction)

        time.sleep(0.05)

    robot.stop()


# ==========================================
# START AUTO
# ==========================================

def start_auto(b):

    global auto_running

    if auto_running:
        return

    auto_running = True

    threading.Thread(
        target=auto_loop,
        daemon=True
    ).start()

    print("AUTO STARTED")


# ==========================================
# STOP AUTO
# ==========================================

def stop_auto(b):

    global auto_running

    auto_running = False

    robot.stop()

    motor_label.value = "Motors: STOPPED"

    print("AUTO STOPPED")


# ==========================================
# BUTTONS
# ==========================================

start_auto_button = widgets.Button(
    description="START AUTO"
)

stop_auto_button = widgets.Button(
    description="STOP AUTO"
)

start_auto_button.on_click(start_auto)
stop_auto_button.on_click(stop_auto)


# ==========================================
# SHOW
# ==========================================

display(camera_view)
display(prediction_label)
display(motor_label)

display(
    widgets.HBox([
        start_auto_button,
        stop_auto_button
    ])
)

Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

Label(value='Steering: ---')

Label(value='Motors: STOPPED')

In [4]:
from jetbot import Robot, bgr8_to_jpeg
import torch
import time
import threading
import ipywidgets as widgets
import traitlets

from PIL import Image
from torchvision import transforms
from IPython.display import display


# ==========================================
# ROBOT
# ==========================================

robot = Robot()


# ==========================================
# AUTO SETTINGS
# ==========================================

AUTO_SPEED = 0.15

LEFT_MOTOR_GAIN = 1.035
RIGHT_MOTOR_GAIN = 1.00

STEERING_SCALE = 0.10
MAX_STEERING = 0.20

auto_running = False


# ==========================================
# IMAGE TRANSFORM
# ==========================================

live_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])


# ==========================================
# LIVE CAMERA
# ==========================================

camera_view = widgets.Image(
    format="jpeg",
    width=400,
    height=300
)

camera_link = traitlets.dlink(
    (camera, "value"),
    (camera_view, "value"),
    transform=bgr8_to_jpeg
)


# ==========================================
# DISPLAY LABELS
# ==========================================

prediction_label = widgets.Label(
    value="Steering: ---"
)

motor_label = widgets.Label(
    value="Motors: STOPPED"
)


# ==========================================
# MOTOR CONTROL
# ==========================================

def drive_from_prediction(prediction):

    steering = max(
        -MAX_STEERING,
        min(MAX_STEERING, prediction)
    )

    steering_power = steering * STEERING_SCALE

    left_speed = (
        AUTO_SPEED + steering_power
    ) * LEFT_MOTOR_GAIN

    right_speed = (
        AUTO_SPEED - steering_power
    ) * RIGHT_MOTOR_GAIN

    left_speed = max(0.0, min(1.0, left_speed))
    right_speed = max(0.0, min(1.0, right_speed))

    robot.left_motor.value = left_speed
    robot.right_motor.value = right_speed

    motor_label.value = (
        f"L: {left_speed:.3f} | "
        f"R: {right_speed:.3f}"
    )


# ==========================================
# AUTO LOOP
# ==========================================

def auto_loop():

    global auto_running

    try:

        while auto_running:

            image_bgr = camera.value

            if image_bgr is None:
                robot.stop()
                time.sleep(0.05)
                continue

            image_rgb = image_bgr[:, :, ::-1].copy()
            pil_image = Image.fromarray(image_rgb)

            input_tensor = (
                live_transform(pil_image)
                .unsqueeze(0)
                .to(device)
            )

            with torch.no_grad():
                steering_prediction = new_model(
                    input_tensor
                ).item()

            prediction_label.value = (
                f"Steering: {steering_prediction:.3f}"
            )

            drive_from_prediction(
                steering_prediction
            )

            time.sleep(0.05)

    except Exception as e:

        print("AUTO ERROR:", e)

    finally:

        robot.stop()
        auto_running = False
        motor_label.value = "Motors: STOPPED"


# ==========================================
# START AUTO
# ==========================================

def start_auto(b):

    global auto_running

    if auto_running:
        return

    auto_running = True

    threading.Thread(
        target=auto_loop,
        daemon=True
    ).start()

    print("AUTO STARTED")


# ==========================================
# STOP AUTO
# ==========================================

def stop_auto(b):

    global auto_running

    auto_running = False

    robot.stop()

    motor_label.value = "Motors: STOPPED"

    print("AUTO STOPPED")


# ==========================================
# BUTTONS
# ==========================================

start_auto_button = widgets.Button(
    description="START AUTO"
)

stop_auto_button = widgets.Button(
    description="STOP AUTO"
)

start_auto_button.on_click(start_auto)
stop_auto_button.on_click(stop_auto)


# ==========================================
# SHOW
# ==========================================

display(camera_view)
display(prediction_label)
display(motor_label)

display(
    widgets.HBox([
        start_auto_button,
        stop_auto_button
    ])
)

Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

Label(value='Steering: ---')

Label(value='Motors: STOPPED')

In [ ]:
from PIL import Image
from torchvision import transforms

live_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

image_bgr = camera.value

image_rgb = image_bgr[:, :, ::-1].copy()
pil_image = Image.fromarray(image_rgb)

input_tensor = (
    live_transform(pil_image)
    .unsqueeze(0)
    .to(device)
)

with torch.no_grad():
    prediction = model(input_tensor).item()

print("Live Prediction:", prediction)

In [6]:
camera.stop()
print("Camera stopped")

Camera stopped


from jetbot import Robot
import time

robot = Robot()

robot.left_motor.value = 0.15
robot.right_motor.value = 0.15

time.sleep(1)

robot.stop()

print("Motor test finished")